# BERT Email Spam Classifier (Local)

In [ ]:
import os
import sys
import re
import tarfile
import urllib.request
from pathlib import Path
from datetime import date

import pandas as pd
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
CORPUS_DIR = DATA_DIR / 'enron6'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
MODEL_DIR = PROJECT_ROOT / 'models'
ARCHIVE_PATH = DATA_DIR / 'enron6.tar.gz'
ARCHIVE_URL = 'http://nlp.cs.aueb.gr/software_and_datasets/Enron-Spam/preprocessed/enron6.tar.gz'

for folder in (DATA_DIR, OUTPUT_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Fetch and extract the corpus

In [ ]:
if not CORPUS_DIR.exists():
    if not ARCHIVE_PATH.exists():
        urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
    with tarfile.open(ARCHIVE_PATH, 'r:gz') as archive:
        archive.extractall(DATA_DIR)

SPAM_DIR = CORPUS_DIR / 'spam'
HAM_DIR = CORPUS_DIR / 'ham'
print(SPAM_DIR.exists(), HAM_DIR.exists())

## Load raw emails

In [ ]:
def collect_text_documents(folder):
    paths = []
    for root, _, files in os.walk(folder):
        for name in files:
            if name.endswith('.txt'):
                paths.append(os.path.join(root, name))
    return paths


spam_files = collect_text_documents(SPAM_DIR)
ham_files = collect_text_documents(HAM_DIR)
print(len(spam_files), len(ham_files))

In [ ]:
def read_documents(paths, label):
    rows = []
    for path in tqdm(paths, file=sys.stdout, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}'):
        with open(path, 'r', encoding='latin-1') as handle:
            rows.append({'content': handle.read(), 'is_spam': label})
    return pd.DataFrame(rows)


data = pd.concat([read_documents(spam_files, 1), read_documents(ham_files, 0)])
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data.to_csv(OUTPUT_DIR / 'data_extracted.csv', index=False, escapechar='\\')
print(data.shape)

## Balance the classes

In [ ]:
class_counts = data['is_spam'].value_counts()
class_counts.to_csv(OUTPUT_DIR / 'class_counts_raw.csv', header=['count'])
print(class_counts)

In [ ]:
spam_rows = data[data['is_spam'] == 1]
ham_rows = data[data['is_spam'] == 0]
spam_downsampled = spam_rows.sample(n=len(ham_rows), random_state=42)

data = pd.concat([spam_downsampled, ham_rows]).sample(frac=1, random_state=42).reset_index(drop=True)
data['is_spam'].value_counts().to_csv(OUTPUT_DIR / 'class_counts_balanced.csv', header=['count'])
data.to_csv(OUTPUT_DIR / 'data_balanced.csv', index=False, escapechar='\\')
print(data['is_spam'].value_counts())

## Normalize the text

In [ ]:
def normalize_email_body(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\W+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


data['content'] = data['content'].apply(normalize_email_body)
data.to_csv(OUTPUT_DIR / 'data_text_processed.csv', index=False, escapechar='\\')
print(data.iloc[0]['content'][:300])

## Split, tokenize, and build the model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['content'], data['is_spam'], test_size=0.2, random_state=42
)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


def encode_batch(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )


train_encodings = encode_batch(X_train.tolist())
test_encodings = encode_batch(X_test.tolist())

y_train = torch.tensor(y_train.values)
y_test = torch.tensor(y_test.values)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(DEVICE)
print(sum(p.numel() for p in model.parameters()))

## Train

In [ ]:
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)

EPOCHS = 3
training_log = []
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in tqdm(train_loader, file=sys.stdout, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}'):
        input_ids, attention_mask, labels = (t.to(DEVICE) for t in batch)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        running_loss += outputs.loss.item()
    epoch_loss = running_loss / len(train_loader)
    training_log.append({'epoch': epoch + 1, 'mean_loss': epoch_loss})
    print(f'Epoch {epoch + 1}/{EPOCHS} mean loss: {epoch_loss:.6f}')

pd.DataFrame(training_log).to_csv(OUTPUT_DIR / 'training_log.csv', index=False)

## Evaluate

In [ ]:
test_dataset = TensorDataset(test_encodings['input_ids'], test_encodings['attention_mask'], y_test)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model.eval()
predictions = []
true_labels = []
with torch.no_grad():
    for batch in tqdm(test_loader, file=sys.stdout, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}'):
        input_ids, attention_mask, labels = (t.to(DEVICE) for t in batch)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)
print(f'Model Accuracy: {accuracy * 100:.2f}%')

In [ ]:
results = pd.DataFrame({'content': X_test.values, 'actual': true_labels, 'predicted': predictions})
results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False, escapechar='\\')

cm = confusion_matrix(true_labels, predictions)
pd.DataFrame(cm, index=['actual_ham', 'actual_spam'], columns=['pred_ham', 'pred_spam']).to_csv(
    OUTPUT_DIR / 'confusion_matrix.csv'
)

report = classification_report(true_labels, predictions, target_names=['ham', 'spam'])
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as handle:
    handle.write(f'accuracy: {accuracy:.4f}\n\n')
    handle.write(report)
print(report)

## Save and reload the model

In [ ]:
stamp = date.today().strftime('%d-%m-%Y')
model_path = MODEL_DIR / f'bert-email-spam-classifier_{stamp}'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(model_path)

In [ ]:
loaded_model = BertForSequenceClassification.from_pretrained(model_path)
loaded_tokenizer = BertTokenizer.from_pretrained(model_path)
loaded_model.to(DEVICE)
loaded_model.eval()

## Classify new messages

In [ ]:
def classify_message(text):
    inputs = loaded_tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.no_grad():
        logits = loaded_model(**inputs).logits
    return 'Spam' if torch.argmax(logits, dim=1).item() == 1 else 'Not Spam'

In [ ]:
samples = {
    'prize_offer': "Congratulations! You've won a free iPhone!",
    'meeting_update': 'Subject: Meeting Re-Scheduled to 3 PM  Hi Everyone! Our meeting is rescheduled by 1 hr late sorry for the inconvenience...',
    'registration_services': '''Subject: Urgent attention required for best work

Urgent attention required for best work
Services:

    Udyam (MSME) Registration
    Import Export Code (IEC) Registration
    FSSAI Registration
    Trade Mark Registration
    PF Registration & Return Filing
    GST Return Filing
    80G & 12A Registration
    NGO DARPAN
    ISO Registration
    Startup India Registration
    CSR Registration

Warm Regards
Name - SUMIT TIWARI
Mobile - 8750906680
''',
    'workday_notice': '''Subject: Workday maintenance window
Hi All,
I would like to inform you about an upcoming Workday maintenance window that will result in temporary system blackouts over the next few days.
''',
}

sample_results = pd.DataFrame(
    [{'name': name, 'text': text, 'prediction': classify_message(text)} for name, text in samples.items()]
)
sample_results.to_csv(OUTPUT_DIR / 'sample_predictions.csv', index=False, escapechar='\\')
print(sample_results[['name', 'prediction']])

## Model footprint

In [ ]:
total_params = sum(p.numel() for p in loaded_model.parameters())
memory_bytes = total_params * 4
footprint = pd.DataFrame(
    [{
        'total_parameters': total_params,
        'memory_bytes': memory_bytes,
        'memory_mb': round(memory_bytes / (1024 ** 2), 2),
        'memory_gb': round(memory_bytes / (1024 ** 3), 3),
    }]
)
footprint.to_csv(OUTPUT_DIR / 'model_footprint.csv', index=False)
print(footprint.T)